In this notebook, we will test an agent that queries a SQL database using natural language by translating the user’s question into SQL. For a given question related to the data in a SQL database, the results of the agent’s generated queries will be compared with those of manually written queries.

In [1]:
%pip install jsonschema
%pip install langchain
%pip install langchain_mistralai
%pip install pandas
%pip install sqlalchemy
%pip install sqlalchemy
%pip install langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


First, we create the agent. For more details, see the sample notebook `single_sql.ipynb`.

In [2]:
import sqlite3
import os
from langchain.tools import tool
import sqlite3
from sqlalchemy import ForeignKey
from sqlalchemy import String, Integer, Boolean, Date
from sqlalchemy.orm import DeclarativeBase
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship
from sqlalchemy import create_engine, MetaData, Table, Column
from sqlalchemy import inspect
from langchain_community.utilities.sql_database import SQLDatabase

sqlite_db="./data/employee.db"
sqlite_db_create="./data/employee.sql"
def init_sqlite3_db():
    try:
        if not os.path.exists(sqlite_db):
            connection = sqlite3.connect(sqlite_db)
            with open(sqlite_db_create, "r") as infile:
              content = infile.read() # Reads entire remaining content as a string
            #print(content)
            connection.executescript(content)
            connection.commit()
            connection.close()
        
    except Exception as e:
        print(f"Error running SQL query: {e}")

def query_sqlite3_db(query: str) -> list[any]:
    conn = sqlite3.connect(sqlite_db)
    c = conn.cursor()
    c.execute(query)
    matches = c.fetchall()
    c.close()
    return matches
    
init_sqlite3_db();


@tool
def query_sql(query: str) -> list[any]:
    """Run a SQL query on the provided data."""
    try:
        return  query_sqlite3_db(query)
    except Exception as e:  
        serr=f"Error running SQL query: {e}"
        return [serr]  


engine = create_engine("sqlite:///data/employee.db", echo=False)
inspector = inspect(engine)
table_names=inspector.get_table_names()
db = SQLDatabase(engine)
sql_schema=db.get_table_info_no_throw([t.strip() for t in table_names])
from langchain.agents import create_agent
from langchain_mistralai.chat_models import ChatMistralAI
import os
import io

template = f"""
Create a SQL query to get the required information from the data.

Here is the schema for the SQL database you will work with:
{sql_schema}

Given an input question, create a syntactically correct SQLite query to run, then look at the results of the query and return the answer.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.
Only use tools if you are sure a query will get a correct answer. 

tools:
query_sql: returns the entities using a given SQL query.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.
Always use double quotes for column names.

Always use the available tools to fetch data.
Only use the results from tools defined above to answer the question.
DO NOT use sample data shown in the table definitions!
DO NOT restrict the number of results with LIMIT clause!
"""

llmm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
    max_retries=2
)


agent = create_agent(
    model=llmm,
    tools=[query_sql],
    system_prompt=template,
)

The result of the agent invocation is the full agent history. The last message in this history is the final answer provided by the agent.

The message immediately before the final answer is a `ToolMessage`, which contains the data fetched by the query from the SQL database. We use this message to compare it with the results of the corresponding manually written SQL query as a reference.

For this purpose, we create a convenience function called `get_answer`. The return value of this function includes the parameter `"answer"`, which contains the query result as JSON string. The `"isOk"` property of the return value indicates whether the tool call was successful.

In [3]:
from langchain.messages import ToolMessage 
from pprint import pprint

def checkAgentCall(output):
    if len(output['messages'])>=4 and type(output['messages'][-2]) is ToolMessage:
        return{
            "firstTry":len(output['messages'])==4,
            "isOk": type(output['messages'][-2]) is ToolMessage,
            "answer": output['messages'][-2].content,
            "query" : output['messages'][-3].tool_calls[0]['args']['query']
        }
    else:
        return {
         "firstTry": True,
         "isOk": False,
         "answer": "",
         "query" : ""
       }
    
def get_answer(question):
    output=agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
    )
    ana=checkAgentCall(output)
    
    value=output["messages"][-1]
    if value != None and hasattr(value, 'content') and value.content is not None:
        if type(value.content) == str:
            ana['modelAnswer']=value.content.strip()
            return ana
        else:
            ana['modelAnswer']=f"Error:{str(value.content)}"
            return ana
    else:
        ana['modelAnswer']=f"No data!"
        return ana

We can now test whether our function works.

In [4]:
get_answer("Please give me the count of employee names.")

{'firstTry': True,
 'isOk': True,
 'answer': '[[201]]',
 'query': 'SELECT COUNT(name) AS employee_count FROM employees;',
 'modelAnswer': 'There are **201** employee names in the database.'}

The answer from the tool call is stored as a JSON string in the `answer` property.

In the file `data/test_queries_v3.json`, you will find the reference query for this question: `SELECT COUNT(*) FROM employees`, which was written manually. Although the generated query is not identical, it produces the same result, so the comparison is straightforward: `answer == answer_ref`.

In other cases, the result is a JSON string containing a row set, represented as an array of arrays. To verify the query results, we created the `compare` function in the Python module `comparesql`.

For more details, see the `comparesql` module.

Our test cases are organized into four files containing JSON data:

- **test_queries_v3.json**: Basic test cases with questions of varying complexity.
- **test_queries_no.json**: None; questions that cannot be answered or yield no result.
- **test_queries_va.json**: Variant; variations of basic questions where the reference result remains the same.
- **test_queries_ex.json**: Extra; additional questions with relatively low complexity.

Each file contains an array of test cases. A test case is structured as follows:

```json
{
    "question": "Please give me the number of employees.",
    "js_query_ref_type": "js",
    "js_query_ref": "var result = data.employees.length;",
    "sql_query_ref": "SELECT COUNT(*) FROM employees",
    "type": "count"
}
```

- **question**: The question about the test data, expressed in natural language.
- **js_query_ref**: The reference JavaScript query to fetch the data.
- **sql_query_ref**: The reference SQL query to fetch the data.
- **type**: The type of query.

For the benchmark, we first use our agent to fetch the data based on the text in `question`. Then, we fetch the data using the SQL query `sql_query_ref`. Both results are compared using the function `compare`. If the comparison is successful, the test case is marked with `answerOK[i] = 1`; otherwise, it is marked with `answerOK[i] = 0`. The comparison results are finally saved in a pandas DataFrame.

In [5]:
import json
import pandas as pd
import comparesql
  
questions=[]
answerOK=[]
category=[]
with open("log_sql.txt", "w", encoding="utf-8") as f:
   f.write("Log\n")

def add_log(line):
   with open("log_sql.txt", "a", encoding="utf-8") as f:
      f.write(str(line)+"\n")

def add_error(cat,question, itry):
   if itry==2:
      category.append(cat)
      questions.append(question)
      answerOK.append(0)

def check_answer_sql(questions, answerOK, qa):
    q=qa["question"]
    sql=qa["sql_query_ref"]
    ref=query_sqlite3_db(sql)
    sref=json.dumps(ref)
    atype=qa["type"]
    add_log(f"Question: {q}")
    answer=get_answer(q)
    add_log(answer)
    add_log('###############')
    add_log(sref)
    iOKAnswer=0
    jref=json.loads(sref)
    janswer=[[]]
    sanswer=answer['answer']
    if len(sanswer):
      janswer=json.loads(answer['answer'])
    copo=False
    if atype=='no-tool-call':
      copo=len(sanswer)==0  
    else:
      copo=comparesql.compare(janswer,jref)
    if copo:
        iOKAnswer=1
    answerOK.append(iOKAnswer)
    questions.append(q)
    add_log("----")


We start with the category basic test cases.

In [6]:
import json
import time
with open('data/test_queries_v3.json') as f:
    testqueries = json.load(f) 
for qa in testqueries['data'][0:25]:
     for ity in range(3):
        try:
          check_answer_sql(questions, answerOK, qa)
          category.append('yes')
          time.sleep(5)
          break
        except Exception as e:
          add_error('yes', qa['question'],  ity)
          add_log(f"Error checking answer: {e}")
          time.sleep(5)
    

2   0


None test cases.

In [7]:
import json
qs_safe=questions.copy()
aok_safe=answerOK.copy()
cat_safe=category.copy()

with open('data/test_queries_no.json') as f:
    testqueries = json.load(f) 

for qa in testqueries['data'][0:25]:
     for ity in range(3):
        try:
          check_answer_sql(questions, answerOK, qa)
          category.append('no')
          time.sleep(5)
          break
        except Exception as e:
          add_error('no', qa['question'],  ity)
          add_log(f"Error checking answer: {e}")
          time.sleep(5)

Variant test cases.

In [8]:
import json
qs_safe=questions.copy()
aok_safe=answerOK.copy()
cat_safe=category.copy()
with open('data/test_queries_va.json') as f:
    testqueries = json.load(f) 
for qa in testqueries['data'][0:25]:
     for ity in range(3):
        try:
          check_answer_sql(questions, answerOK, qa)
          category.append('va')
          time.sleep(5)
          break
        except Exception as e:
          add_error('no', qa['question'],  ity)
          add_log(f"Error checking answer: {e}")
          time.sleep(5)

Retrying langchain_mistralai.chat_models.ChatMistralAI.completion_with_retry.<locals>._completion_with_retry in 4 seconds as it raised ReadTimeout: The read operation timed out.
Retrying langchain_mistralai.chat_models.ChatMistralAI.completion_with_retry.<locals>._completion_with_retry in 4 seconds as it raised ReadTimeout: The read operation timed out.
Retrying langchain_mistralai.chat_models.ChatMistralAI.completion_with_retry.<locals>._completion_with_retry in 4 seconds as it raised ReadTimeout: The read operation timed out.


2   0


Extra test cases.

In [9]:
qs=questions.copy()
aok=answerOK.copy()
cat=category.copy()

with open('data/test_queries_ex.json') as f:
    testqueries = json.load(f) 

test_data=testqueries['data']

for qa in testqueries['data'][0:25]:
     for ity in range(3):
        try:
          check_answer_sql(questions, answerOK, qa)
          category.append('ex')
          time.sleep(5)
          break
        except Exception as e:
          add_error('no', qa['question'],  ity)
          add_log(f"Error checking answer: {e}")
          time.sleep(5)


Retrying langchain_mistralai.chat_models.ChatMistralAI.completion_with_retry.<locals>._completion_with_retry in 4 seconds as it raised ReadTimeout: The read operation timed out.


Now that we have the results in the `questions`, `answerOK`, and `category` lists, we can create a DataFrame and save it for future reference.

In [12]:
df = pd.DataFrame({
  "questions":questions,
  "okAnswer": answerOK  ,
  "category": category  
}                 
)
df.to_pickle("data/result_sql.pkl")


For a comparison with the corresponding SQL queries, see the notebook `compare_sql_js.ipynb`. Here, we will only provide the percentage of correct answers.

In [11]:
df['okAnswer'].sum()/len(df['okAnswer'])*100

np.float64(81.0)